In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Train_trimmed.txt to Train_trimmed.txt
Saving Validation_trimmed_405k.txt to Validation_trimmed_405k.txt
Saving Test_trimmed_405k.txt to Test_trimmed_405k.txt


In [ ]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn pandas numpy torch

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA available: True
GPU: Tesla T4


In [ ]:
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"


LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
REV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)

print(f"Classification Task: {NUM_LABELS}-class sentiment classification")
print(f"Classes: {list(LABEL_MAP.keys())}")
print(f"Label mapping: {LABEL_MAP}")
MAX_LENGTH = 96


NUM_EPOCHS = 10
BATCH_SIZE = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01



DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'Train_trimmed.txt'
VAL_PATH = DATA_DIR / 'Validation_trimmed_405k.txt'
TEST_PATH = DATA_DIR / 'Test_trimmed_405k.txt'

print(f"Train file exists: {TRAIN_PATH.exists()}")
print(f"Validation file exists: {VAL_PATH.exists()}")
print(f"Test file exists: {TEST_PATH.exists()}")

Classification Task: 3-class sentiment classification
Classes: ['negative', 'neutral', 'positive']
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Train file exists: True
Validation file exists: True
Test file exists: True


In [ ]:
def parse_conll_file(path: Path, has_labels: bool = True) -> pd.DataFrame:
    """Parse CoNLL-like format into DataFrame."""
    records = []
    current_id = None
    current_label = None
    current_tokens = []

    def flush_current():
        if current_id is None or not current_tokens:
            return
        text = ' '.join(current_tokens).strip()
        records.append({'id': current_id, 'text': text, 'label_str': current_label})

    with open(path, encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.rstrip('\n')
            if not line.strip():
                flush_current()
                current_id = None
                current_label = None
                current_tokens = []
                continue

            if line.startswith('meta'):
                parts = line.split()
                current_id = parts[1]
                if len(parts) > 2 and has_labels:
                    current_label = parts[2].lower()
                elif len(parts) > 2:
                    current_label = parts[2].lower()
                else:
                    current_label = None
                continue

            token = line.split()[0]
            current_tokens.append(token)

    flush_current()
    df = pd.DataFrame(records)
    if 'label_str' not in df.columns:
        df['label_str'] = None
    return df

In [ ]:
URL_PATTERN = re.compile(r'(https?://\S+|www\.\S+)', re.IGNORECASE)
USER_PATTERN = re.compile(r'@\w+')
MULTISPACE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    """Clean code-mixed text: lowercase, replace URLs/mentions, normalize spaces."""
    text = text.lower()
    text = URL_PATTERN.sub('URL', text)
    text = USER_PATTERN.sub('USER', text)
    text = MULTISPACE.sub(' ', text)
    return text.strip()

In [ ]:
train_df = parse_conll_file(TRAIN_PATH, has_labels=True)
val_df = parse_conll_file(VAL_PATH, has_labels=True)
test_df = parse_conll_file(TEST_PATH, has_labels=False)


for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    df['clean_text'] = df['text'].apply(clean_text)
    if 'label_str' in df.columns:
        df['label_str'] = df['label_str'].str.lower()
        df['label'] = df['label_str'].map(LABEL_MAP)

train_df = train_df[train_df['label'].notna()].reset_index(drop=True)
val_df = val_df[val_df['label'].notna()].reset_index(drop=True)

print('Train label distribution:')
print(train_df['label_str'].value_counts())
print('\nValidation label distribution:')
print(val_df['label_str'].value_counts())
print('\nDataset sizes -> train:', len(train_df), '| val:', len(val_df), '| test:', len(test_df))
train_df.head()

Train label distribution:
label_str
neutral     4354
positive    4041
negative    3451
Name: count, dtype: int64

Validation label distribution:
label_str
neutral     617
positive    556
negative    494
Name: count, dtype: int64

Dataset sizes -> train: 11846 | val: 1667 | test: 1713


,id,text,label_str,clean_text,label
0,3,@ AdilNisarButt pakistan ka ghra tauq he Pakis...,negative,@ adilnisarbutt pakistan ka ghra tauq he pakis...,0
1,41,Madarchod mulle ye mathura me Nahi dikha tha j...,negative,madarchod mulle ye mathura me nahi dikha tha j...,0
2,48,@ narendramodi Manya Pradhan Mantri mahoday Sh...,positive,@ narendramodi manya pradhan mantri mahoday sh...,2
3,64,@ Atheist _ Krishna Jcb full trend me chal rah...,positive,@ atheist _ krishna jcb full trend me chal rah...,2
4,66,@ AbhisharSharma _ @ RavishKumarBlog Loksabha ...,positive,@ abhisharsharma _ @ ravishkumarblog loksabha ...,2


In [ ]:
class SentimentDataset(Dataset):
    """PyTorch Dataset for tokenized tweets."""
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.encodings = tokenizer(
            list(texts),
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = None if labels is None else [int(label) for label in labels]

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def build_dataset(df: pd.DataFrame, tokenizer, max_length=MAX_LENGTH, use_labels: bool = True):
    """Build dataset from DataFrame."""
    labels = None
    if use_labels and 'label' in df.columns:
        labels = df['label'].tolist()
    return SentimentDataset(df['clean_text'].tolist(), labels, tokenizer, max_length=max_length)

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy and macro F1 for evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

summary_rows = []

def train_and_evaluate(model_name: str,
                       train_df: pd.DataFrame,
                       val_df: pd.DataFrame,
                       test_df: pd.DataFrame,
                       max_length: int = 96,
                       num_epochs: int = 3,
                       learning_rate: float = 2e-5,
                       weight_decay: float = 0.01):
    """Train and evaluate a model."""
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}\n")

    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
        num_labels = globals().get('NUM_LABELS', 3)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels, token=hf_token
        )
    except Exception as e:
        print(f"Error loading {model_name}: {e}")
        print("Skipping this model...")
        return None, None, None

    train_dataset = build_dataset(train_df, tokenizer, max_length=max_length, use_labels=True)
    val_dataset = build_dataset(val_df, tokenizer, max_length=max_length, use_labels=True)
    test_dataset = build_dataset(test_df, tokenizer, max_length=max_length, use_labels=False)

    batch_size = globals().get('BATCH_SIZE', 8)
    seed = globals().get('SEED', 42)
    num_labels = globals().get('NUM_LABELS', 3)

    output_dir = f"./{model_name.replace('/', '_')}_outputs"
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        weight_decay=weight_decay,
        logging_steps=50,
        report_to='none',
        seed=seed,
        push_to_hub=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    print("Starting training...")
    trainer.train()

    eval_metrics = trainer.evaluate()
    print(f"\nValidation Metrics: {eval_metrics}")

    val_predictions = trainer.predict(val_dataset)
    val_pred_labels = np.argmax(val_predictions.predictions, axis=-1)
    print('\n' + '='*60)
    print(f'Validation Classification Report for {model_name}')
    print('='*60)
    print(classification_report(
        val_df['label'], val_pred_labels,
        target_names=['negative', 'neutral', 'positive'],
        digits=4
    ))

    print('\nPer-Class Accuracy Breakdown:')
    print('-'*60)
    for i, class_name in enumerate(['negative', 'neutral', 'positive']):
        class_mask = val_df['label'] == i
        if class_mask.sum() > 0:
            class_acc = accuracy_score(
                val_df.loc[class_mask, 'label'],
                val_pred_labels[class_mask]
            )
            print(f"{class_name.capitalize():<15}: {class_acc*100:.2f}% ({class_mask.sum()} samples)")
    print('-'*60)

    test_predictions = trainer.predict(test_dataset)
    test_pred_labels = np.argmax(test_predictions.predictions, axis=-1)

    pred_df = test_df.copy()
    pred_df['pred_label'] = test_pred_labels
    rev_label_map = globals().get('REV_LABEL_MAP', {0: 'negative', 1: 'neutral', 2: 'positive'})
    pred_df['pred_sentiment'] = pred_df['pred_label'].map(rev_label_map)

    prediction_file = f"TEST_predictions_{model_name.replace('/', '_')}.csv"
    pred_columns = [col for col in ['id', 'text', 'clean_text', 'pred_label', 'pred_sentiment']
                     if col in pred_df.columns]
    pred_df[pred_columns].to_csv(prediction_file, index=False)
    print(f'\nSaved test predictions to {prediction_file}')

    summary_rows.append({
        'Model': model_name,
        'Val Accuracy': eval_metrics.get('eval_accuracy', 0),
        'Val Macro-F1': eval_metrics.get('eval_f1_macro', 0),
        'Predictions File': prediction_file
    })

    return trainer, eval_metrics, prediction_file

In [ ]:
xlm_trainer, xlm_metrics, xlm_pred_file = train_and_evaluate(
    model_name='xlm-roberta-base',
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)


Training xlm-roberta-base



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3518279288.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.124600,1.104834,0.370126,0.180093
2,1.083800,1.097331,0.470306,0.375347
3,0.936100,0.959974,0.543491,0.543019



Validation Metrics: {'eval_loss': 0.9599736332893372, 'eval_accuracy': 0.5434913017396521, 'eval_f1_macro': 0.5430185138304128, 'eval_runtime': 10.607, 'eval_samples_per_second': 157.16, 'eval_steps_per_second': 78.627, 'epoch': 3.0}

Validation Classification Report for xlm-roberta-base
              precision    recall  f1-score   support

    negative     0.5101    0.7186    0.5966       494
     neutral     0.4568    0.3517    0.3974       617
    positive     0.6734    0.6007    0.6350       556

    accuracy                         0.5435      1667
   macro avg     0.5468    0.5570    0.5430      1667
weighted avg     0.5448    0.5435    0.5357      1667


Per-Class Accuracy Breakdown:
------------------------------------------------------------
Negative       : 71.86% (494 samples)
Neutral        : 35.17% (617 samples)
Positive       : 60.07% (556 samples)
------------------------------------------------------------



Saved test predictions to TEST_predictions_xlm-roberta-base.csv


In [ ]:
mbert_trainer, mbert_metrics, mbert_pred_file = train_and_evaluate(
    model_name='bert-base-multilingual-cased',
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)


Training bert-base-multilingual-cased



tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3518279288.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.106800,1.124182,0.296341,0.152399
2,1.032300,1.040252,0.452909,0.359960
3,0.951800,0.956390,0.568686,0.572857



Validation Metrics: {'eval_loss': 0.956390380859375, 'eval_accuracy': 0.5686862627474505, 'eval_f1_macro': 0.5728574334812288, 'eval_runtime': 10.7071, 'eval_samples_per_second': 155.69, 'eval_steps_per_second': 77.892, 'epoch': 3.0}

Validation Classification Report for bert-base-multilingual-cased
              precision    recall  f1-score   support

    negative     0.5514    0.5972    0.5734       494
     neutral     0.4870    0.4862    0.4866       617
    positive     0.6841    0.6349    0.6586       556

    accuracy                         0.5687      1667
   macro avg     0.5742    0.5728    0.5729      1667
weighted avg     0.5718    0.5687    0.5697      1667


Per-Class Accuracy Breakdown:
------------------------------------------------------------
Negative       : 59.72% (494 samples)
Neutral        : 48.62% (617 samples)
Positive       : 63.49% (556 samples)
------------------------------------------------------------



Saved test predictions to TEST_predictions_bert-base-multilingual-cased.csv


In [ ]:
summary_df = pd.DataFrame(summary_rows)

print('\n' + '='*70)
print('FINAL RESULTS SUMMARY - 3-Class Sentiment Classification')
print('='*70)
print('\nClassification Classes: negative, neutral, positive\n')

xlm_acc = None
mbert_acc = None
xlm_f1 = None
mbert_f1 = None

for _, row in summary_df.iterrows():
    model_name = row['Model']
    if 'xlm' in model_name.lower() or 'roberta' in model_name.lower():
        if not pd.isna(row['Val Accuracy']):
            xlm_acc = row['Val Accuracy']
            xlm_f1 = row['Val Macro-F1']
    elif 'bert' in model_name.lower() and 'xlm' not in model_name.lower():
        if not pd.isna(row['Val Accuracy']):
            mbert_acc = row['Val Accuracy']
            mbert_f1 = row['Val Macro-F1']

# ======================================================================
# PART 1: INDIVIDUAL MODEL ACCURACIES
# ======================================================================
print('='*70)
print('INDIVIDUAL MODEL ACCURACIES')
print('='*70)

print('\n1. XLM-RoBERTa Model Results:')
print('   ' + '-'*65)
if xlm_acc is not None:
    print(f'   Validation Accuracy: {xlm_acc*100:.0f}%')
    print(f'   Macro F1-Score:       {xlm_f1*100:.0f}%')
    print(f'   Status:               Training completed successfully')
else:
    print('   Validation Accuracy: N/A')
    print('   Macro F1-Score:       N/A')
    print('   Status:               Training failed or not completed')

print('\n2. mBERT (Bilingual BERT) Model Results:')
print('   ' + '-'*65)
if mbert_acc is not None:
    print(f'   Validation Accuracy: {mbert_acc*100:.0f}%')
    print(f'   Macro F1-Score:       {mbert_f1*100:.0f}%')
    print(f'   Status:               Training completed successfully')
else:
    print('   Validation Accuracy: N/A')
    print('   Macro F1-Score:       N/A')
    print('   Status:               Training failed or not completed')

# ======================================================================
# PART 2: COMPARISON
# ======================================================================
print('\n' + '='*70)
print('MODEL COMPARISON')
print('='*70)

print('\nComparison Table:')
print('-'*70)
print(f"{'Model':<40} {'Validation Accuracy':<25} {'Macro F1-Score':<20}")
print('-'*70)

for _, row in summary_df.iterrows():
    model_name = row['Model']
    if pd.isna(row['Val Accuracy']) or pd.isna(row['Val Macro-F1']):
        print(f"{model_name:<40} {'N/A':<25} {'N/A':<20}")
    else:
        acc = row['Val Accuracy']
        f1 = row['Val Macro-F1']
        acc_pct = f"{acc*100:.0f}%"
        f1_pct = f"{f1*100:.0f}%"
        print(f"{model_name:<40} {acc_pct:<25} {f1_pct:<20}")

print('-'*70)

if xlm_acc is not None and mbert_acc is not None:
    print('\nWinner:')
    print('-'*70)
    if xlm_acc > mbert_acc:
        diff = (xlm_acc - mbert_acc) * 100
        print(f'✓ XLM-RoBERTa performs better')
        print(f'  Accuracy: {xlm_acc*100:.0f}% (mBERT: {mbert_acc*100:.0f}%)')
        print(f'  Difference: +{diff:.0f} percentage points')
    elif mbert_acc > xlm_acc:
        diff = (mbert_acc - xlm_acc) * 100
        print(f'✓ mBERT (Bilingual BERT) performs better')
        print(f'  Accuracy: {mbert_acc*100:.0f}% (XLM-RoBERTa: {xlm_acc*100:.0f}%)')
        print(f'  Difference: +{diff:.0f} percentage points')
    else:
        print(f'Both models achieved the same accuracy: {xlm_acc*100:.0f}%')
    print('-'*70)
elif xlm_acc is not None:
    print('\nNote: Only XLM-RoBERTa results available for comparison')
elif mbert_acc is not None:
    print('\nNote: Only mBERT results available for comparison')
else:
    print('\nNote: No model results available for comparison')

print('\nPER-CLASS PERFORMANCE:')
print('='*70)

if 'xlm_trainer' in globals() and xlm_trainer is not None:
    print('\nXLM-RoBERTa Classification Report:')
    print('-'*70)
    try:
        from transformers import AutoTokenizer
        xlm_tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
        max_len = globals().get('MAX_LENGTH', 96)
        val_dataset_xlm = build_dataset(val_df, xlm_tokenizer, max_length=max_len, use_labels=True)
        val_pred_xlm = xlm_trainer.predict(val_dataset_xlm)
        val_pred_labels_xlm = np.argmax(val_pred_xlm.predictions, axis=-1)
        print(classification_report(
            val_df['label'], val_pred_labels_xlm,
            target_names=['negative', 'neutral', 'positive'],
            digits=4
        ))
    except Exception as e:
        print(f"Could not generate detailed report: {e}")
        print("(Summary metrics are still available above)")

if 'mbert_trainer' in globals() and mbert_trainer is not None:
    print('\nmBERT Classification Report:')
    print('-'*70)
    try:
        mbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
        max_len = globals().get('MAX_LENGTH', 96)
        val_dataset_mbert = build_dataset(val_df, mbert_tokenizer, max_length=max_len, use_labels=True)
        val_pred_mbert = mbert_trainer.predict(val_dataset_mbert)
        val_pred_labels_mbert = np.argmax(val_pred_mbert.predictions, axis=-1)
        print(classification_report(
            val_df['label'], val_pred_labels_mbert,
            target_names=['negative', 'neutral', 'positive'],
            digits=4
        ))
    except Exception as e:
        print(f"Could not generate detailed report: {e}")
        print("(Summary metrics are still available above)")

print('\n' + '='*70)
print('Prediction files saved:')
for row in summary_rows:
    print(f"  - {row['Model']}: {row['Predictions File']}")
print('='*70)


FINAL RESULTS SUMMARY - 3-Class Sentiment Classification

Classification Classes: negative, neutral, positive

INDIVIDUAL MODEL ACCURACIES

1. XLM-RoBERTa Model Results:
   -----------------------------------------------------------------
   Validation Accuracy: 54%
   Macro F1-Score:       54%
   Status:               Training completed successfully

2. mBERT (Bilingual BERT) Model Results:
   -----------------------------------------------------------------
   Validation Accuracy: 57%
   Macro F1-Score:       57%
   Status:               Training completed successfully

MODEL COMPARISON

Comparison Table:
----------------------------------------------------------------------
Model                                    Validation Accuracy       Macro F1-Score      
----------------------------------------------------------------------
xlm-roberta-base                         54%                       54%                 
bert-base-multilingual-cased             57%                      

              precision    recall  f1-score   support

    negative     0.5101    0.7186    0.5966       494
     neutral     0.4568    0.3517    0.3974       617
    positive     0.6734    0.6007    0.6350       556

    accuracy                         0.5435      1667
   macro avg     0.5468    0.5570    0.5430      1667
weighted avg     0.5448    0.5435    0.5357      1667


mBERT Classification Report:
----------------------------------------------------------------------


              precision    recall  f1-score   support

    negative     0.5514    0.5972    0.5734       494
     neutral     0.4870    0.4862    0.4866       617
    positive     0.6841    0.6349    0.6586       556

    accuracy                         0.5687      1667
   macro avg     0.5742    0.5728    0.5729      1667
weighted avg     0.5718    0.5687    0.5697      1667


Prediction files saved:
  - xlm-roberta-base: TEST_predictions_xlm-roberta-base.csv
  - bert-base-multilingual-cased: TEST_predictions_bert-base-multilingual-cased.csv


In [ ]:
from google.colab import files
files.download('TEST_predictions_bert-base-multilingual-cased.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

CLASSES = ['negative', 'neutral', 'positive']
try:
    pred_df_xlm = pd.read_csv('TEST_predictions_xlm-roberta-base.csv')
    print("="*70)
    print("XLM-RoBERTa Test Predictions")
    print("="*70)
    print(f"Total predictions: {len(pred_df_xlm)}")
    print("\nPrediction distribution (3 classes):")
    print(pred_df_xlm['pred_sentiment'].value_counts().reindex(['negative','neutral','positive'], fill_value=0)
)
    print("\nSample predictions:")
    print(pred_df_xlm[['id', 'text', 'pred_sentiment']].head(10))
except FileNotFoundError:
    print("XLM-RoBERTa predictions file not found")

try:
    pred_df_mbert = pd.read_csv('TEST_predictions_bert-base-multilingual-cased.csv')
    print("\n" + "="*70)
    print("mBERT Test Predictions")
    print("="*70)
    print(f"Total predictions: {len(pred_df_mbert)}")
    print("\nPrediction distribution (3 classes):")
    print(pred_df_mbert['pred_sentiment'].value_counts().reindex(CLASSES, fill_value=0))
    print("\nSample predictions:")
    print(pred_df_mbert[['id', 'text', 'pred_sentiment']].head(10))
except FileNotFoundError:
    print("mBERT predictions file not found")

if 'pred_df_xlm' in locals() and 'pred_df_mbert' in locals():
    print("\n" + "="*70)
    print("Model Comparison: Agreement on Predictions")
    print("="*70)
    merged = pred_df_xlm.merge(pred_df_mbert, on='id', suffixes=('_xlm', '_mbert'))
    agreement = (merged['pred_sentiment_xlm'] == merged['pred_sentiment_mbert']).sum()
    total = len(merged)
    print(f"Agreement: {agreement}/{total} ({agreement/total*100:.2f}%)")
    print(f"Disagreement: {total-agreement}/{total} ({(total-agreement)/total*100:.2f}%)")

XLM-RoBERTa predictions file not found
mBERT predictions file not found
